In [1]:
import torch
print(torch.__version__)

2.6.0+cu124


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"   # idle GPU

import itertools, json, math, re, sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional, Dict

import pdfplumber, spacy, torch, pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM,
    GPT2LMHeadModel, GPT2TokenizerFast,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

QG_MODEL_NAME = "valhalla/t5-base-qg-hl"
QA2D_MODEL_NAME = "MarkS/bart-base-qa2d"
FILL_MODEL_NAME = "bert-large-uncased"
BRIDGE_ENT_LABELS = {"PERSON", "ORG", "GPE", "WORK_OF_ART", "EVENT", "PRODUCT", "LAW"}

/home/UFAD/akashbalaji/.conda/envs/mhqg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [3]:
"""
Cross-Document Bridge Multi-Hop Question Generation (folder mode)
===================================================================
Same method as before (Pan et al. 2021 bridge-question style), but now:
  - takes a FOLDER of PDFs belonging to one topic (e.g. "characterization/")
  - generates bridge questions for every PAIRWISE combination of PDFs in
    that folder (3 PDFs -> 3 pairs, 2 PDFs -> 1 pair)
  - tags every generated question with which two source PDFs it came from

Run from Jupyter with the `mhqg` conda kernel.
"""

import itertools
import json
import re
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional, Dict

import pdfplumber
import spacy
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QG_MODEL_NAME = "valhalla/t5-base-qg-hl"
QA2D_MODEL_NAME = "MarkS/bart-base-qa2d"
FILL_MODEL_NAME = "bert-large-uncased"

BRIDGE_ENT_LABELS = {"PERSON", "ORG", "GPE", "WORK_OF_ART", "EVENT", "PRODUCT", "LAW"}


# --------------------------------------------------------------------------
# PDF -> text -> sentences -> entities  (per file, cached in a dict)
# --------------------------------------------------------------------------
def extract_text(pdf_path: Path) -> str:
    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            chunks.append(page.extract_text() or "")
    text = "\n".join(chunks)
    text = re.sub(r"-\n", "", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


def split_sentences(nlp, text: str) -> List[str]:
    doc = nlp(text)
    return [s.text.strip() for s in doc.sents if len(s.text.strip()) > 20]


@dataclass
class EntMention:
    text: str
    label: str
    sent: str


def extract_entities(nlp, sentences: List[str]) -> List[EntMention]:
    out = []
    for sent in sentences:
        doc = nlp(sent)
        for ent in doc.ents:
            if ent.label_ in BRIDGE_ENT_LABELS:
                out.append(EntMention(text=ent.text.strip(), label=ent.label_, sent=sent))
    return out


@dataclass
class DocIndex:
    filename: str
    sentences: List[str]
    entities: List[EntMention]


def index_folder(nlp, folder: Path) -> Dict[str, DocIndex]:
    """Extract + NER-index every PDF in a folder. Returns {filename: DocIndex}."""
    docs = {}
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        raise FileNotFoundError(f"No PDFs found in {folder}")
    for pdf_path in pdfs:
        print(f"  [index] {pdf_path.name}", file=sys.stderr)
        text = extract_text(pdf_path)
        sents = split_sentences(nlp, text)
        ents = extract_entities(nlp, sents)
        docs[pdf_path.name] = DocIndex(filename=pdf_path.name, sentences=sents, entities=ents)
    return docs


# --------------------------------------------------------------------------
# Bridge entity discovery between two indexed docs
# --------------------------------------------------------------------------
def find_bridge_entities(doc_a: DocIndex, doc_b: DocIndex) -> List[str]:
    norm = lambda s: s.lower().strip()
    set_a = {norm(e.text) for e in doc_a.entities}
    set_b = {norm(e.text) for e in doc_b.entities}
    common = set_a & set_b
    return sorted([c for c in common if len(c) > 2])


def first_sentence_with_entity(mentions: List[EntMention], entity_norm: str) -> Optional[EntMention]:
    for m in mentions:
        if m.text.lower().strip() == entity_norm:
            return m
    return None


def pick_answer_entity(mentions: List[EntMention], sent: str, exclude_norm: str) -> Optional[str]:
    for m in mentions:
        if m.sent == sent and m.text.lower().strip() != exclude_norm:
            return m.text
    return None


# --------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------
class QuestionGenerator:
    def __init__(self, model_name: str = QG_MODEL_NAME):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()

    def generate(self, context: str, answer: str) -> str:
        if answer not in context:
            context = f"{answer}. {context}"
        highlighted = context.replace(answer, f"<hl> {answer} <hl>", 1)
        inputs = self.tok(f"generate question: {highlighted}", return_tensors="pt",
                           truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_length=64, num_beams=4)
        return self.tok.decode(out[0], skip_special_tokens=True).strip()


# class QA2D:
#     def __init__(self, model_name: str = QA2D_MODEL_NAME):
#         self.tok = AutoTokenizer.from_pretrained(model_name)
#         self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()

#     def to_declarative(self, question: str, answer: str) -> str:
#         inputs = self.tok(f"{question} [SEP] {answer}", return_tensors="pt",
#                            truncation=True, max_length=128).to(DEVICE)
#         with torch.no_grad():
#             out = self.model.generate(**inputs, max_length=64, num_beams=4)
#         return self.tok.decode(out[0], skip_special_tokens=True).strip()


class QA2D:
    def __init__(self, model_name: str = QA2D_MODEL_NAME):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()

    def to_declarative(self, question: str, answer: str) -> str:
        input_text = f"question: {question} answer: {answer}"
        inputs = self.tok(input_text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_length=64, num_beams=4)
        return self.tok.decode(out[0], skip_special_tokens=True).strip()


class MaskFiller:
    def __init__(self, model_name: str = FILL_MODEL_NAME):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(DEVICE).eval()
        self.mask_token = self.tok.mask_token

    def fuse(self, bridge_entity: str, declarative_sentence: str) -> str:
        masked_clause = declarative_sentence.replace(bridge_entity, self.mask_token, 1) \
            if bridge_entity in declarative_sentence else f"{self.mask_token} {declarative_sentence}"
        prompt = f"The {self.mask_token} that {masked_clause}"
        inputs = self.tok(prompt, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        mask_idx = (inputs["input_ids"][0] == self.tok.mask_token_id).nonzero(as_tuple=True)[0]
        if len(mask_idx) == 0:
            return f"the {bridge_entity} that {declarative_sentence}"
        with torch.no_grad():
            logits = self.model(**inputs).logits
        top_id = logits[0, mask_idx[0]].argmax(dim=-1)
        filled_token = self.tok.decode([top_id]).strip()
        rest = declarative_sentence.split(bridge_entity, 1)[-1] if bridge_entity in declarative_sentence else declarative_sentence
        return f"the {filled_token} that {bridge_entity}{rest}"


# --------------------------------------------------------------------------
# Bridge question construction for one PDF pair
# --------------------------------------------------------------------------
@dataclass
class BridgeQuestionRecord:
    topic: str
    source_pdf_a: str
    source_pdf_b: str
    bridge_entity: str
    answer_entity: str
    question: str
    answer: str
    context_a: str
    context_b: str


def build_bridge_questions_for_pair(
    topic: str, doc_a: DocIndex, doc_b: DocIndex,
    qg: QuestionGenerator, qa2d: QA2D, filler: MaskFiller,
    max_questions: int = 10,
) -> List[BridgeQuestionRecord]:

    bridges = find_bridge_entities(doc_a, doc_b)
    print(f"    [{doc_a.filename} <-> {doc_b.filename}] {len(bridges)} candidate bridge entities", file=sys.stderr)

    records = []
    for bridge in bridges:
        if len(records) >= max_questions:
            break
        mention_a = first_sentence_with_entity(doc_a.entities, bridge)
        mention_b = first_sentence_with_entity(doc_b.entities, bridge)
        if not mention_a or not mention_b:
            continue
        answer_entity = pick_answer_entity(doc_a.entities, mention_a.sent, bridge)
        if not answer_entity:
            continue
        try:
            q_a = qg.generate(mention_a.sent, answer_entity)
            q_b = qg.generate(mention_b.sent, mention_b.text)
            decl_b = qa2d.to_declarative(q_b, mention_b.text)
            fused_clause = filler.fuse(mention_b.text, decl_b)

            if bridge in q_a.lower():
                pattern = re.compile(re.escape(mention_a.text), re.IGNORECASE)
                final_question = pattern.sub(fused_clause, q_a, count=1)
            else:
                final_question = f"{q_a.rstrip('?')}, where {fused_clause}?"

            records.append(BridgeQuestionRecord(
                topic=topic,
                source_pdf_a=doc_a.filename,
                source_pdf_b=doc_b.filename,
                bridge_entity=mention_b.text,
                answer_entity=answer_entity,
                question=final_question,
                answer=answer_entity,
                context_a=mention_a.sent,
                context_b=mention_b.sent,
            ))
        except Exception as e:
            print(f"    [warn] skipped bridge='{bridge}': {e}", file=sys.stderr)
            continue
    return records


# --------------------------------------------------------------------------
# Folder-level driver: pairwise across all PDFs in a topic folder
# --------------------------------------------------------------------------
def run_on_topic_folder(
    topic_folder: Path, nlp, qg: QuestionGenerator, qa2d: QA2D, filler: MaskFiller,
    max_questions_per_pair: int = 10,
) -> List[BridgeQuestionRecord]:

    topic = topic_folder.name
    print(f"[topic] {topic}", file=sys.stderr)
    docs = index_folder(nlp, topic_folder)

    all_records = []
    for name_a, name_b in itertools.combinations(docs.keys(), 2):
        recs = build_bridge_questions_for_pair(
            topic, docs[name_a], docs[name_b], qg, qa2d, filler,
            max_questions=max_questions_per_pair,
        )
        all_records.extend(recs)
    return all_records


def save_jsonl(records: List[BridgeQuestionRecord], out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(asdict(r), ensure_ascii=False) + "\n")
    print(f"[done] wrote {len(records)} records -> {out_path}", file=sys.stderr)

In [4]:
nlp = spacy.load("en_core_web_sm")
qg = QuestionGenerator()
qa2d = QA2D()
filler = MaskFiller()
print("Models loaded.")

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 394/394 [00:00<00:00, 9196.65it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-large-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded.


In [5]:
all_records_by_topic = {}
for t in ["Characterization"]:
    recs = run_on_topic_folder(Path("/home/UFAD/akashbalaji/SCAN/MultiHopQA/Input") / t, nlp, qg, qa2d, filler, max_questions_per_pair=10)
    save_jsonl(recs, Path("/home/UFAD/akashbalaji/SCAN/MultiHopQA/output") / t / "bridge_questions.jsonl")
    all_records_by_topic[t] = recs

[topic] Characterization
  [index] A Novel Approach to Photonic Packaging Leveraging.pdf
  [index] Photonic plug for scalable silicon photonics packaging.pdf
    [A Novel Approach to Photonic Packaging Leveraging.pdf <-> Photonic plug for scalable silicon photonics packaging.pdf] 9 candidate bridge entities
[done] wrote 7 records -> /home/UFAD/akashbalaji/SCAN/MultiHopQA/output/Characterization/bridge_questions.jsonl


In [6]:
dfs = [pd.read_json(Path("/home/UFAD/akashbalaji/SCAN/MultiHopQA/output") / t / "bridge_questions.jsonl", lines=True) for t in ["Characterization"]]
all_df = pd.concat(dfs, ignore_index=True)
all_df[["topic", "source_pdf_a", "source_pdf_b", "question", "answer"]]

,topic,source_pdf_a,source_pdf_b,question,answer
0,Characterization,A Novel Approach to Photonic Packaging Leverag...,Photonic plug for scalable silicon photonics p...,What advanced the fact that CMOS . technologie...,Silicon– Germaniumhetero-junctionbipolartransi...
1,Characterization,A Novel Approach to Photonic Packaging Leverag...,Photonic plug for scalable silicon photonics p...,Where was the the event that Electron Devices ...,"Washington,DC"
2,Characterization,A Novel Approach to Photonic Packaging Leverag...,Photonic plug for scalable silicon photonics p...,What degree did AlexanderJanta-Polczynski rece...,AlexanderJanta-PolczynskireceivedtheB.Eng.degr...
3,Characterization,A Novel Approach to Photonic Packaging Leverag...,Photonic plug for scalable silicon photonics p...,Who is a Member of the journal that IEEE publi...,YoichiTaira
4,Characterization,A Novel Approach to Photonic Packaging Leverag...,Photonic plug for scalable silicon photonics p...,What was the title of the issue of the fact th...,",vol.35,pp.2526–2528,2010"
5,Characterization,A Novel Approach to Photonic Packaging Leverag...,Photonic plug for scalable silicon photonics p...,"Photonicintegration reducesthenumber of who, w...",T. Barwicz
6,Characterization,A Novel Approach to Photonic Packaging Leverag...,Photonic plug for scalable silicon photonics p...,"Who wrote ""An integrated silicon photonics tec...",N. B. Feilchenfeld
